# 5.5 Factor Graphs

Factor graphs expose the algebra of a model: variables are connected exactly to the factors that mention them.

Factor graphs unify directed and undirected factors. They are the cleanest language for belief propagation, junction trees, and message-passing views of probabilistic programs.

Save a copy to Drive to edit.

In [ ]:

import itertools
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5505)


def enumerate_factor_graph(var_sizes, factors):
    variables = list(range(len(var_sizes)))
    assignments = list(itertools.product(*[range(size) for size in var_sizes]))
    scores = []
    for assignment in assignments:
        score = 1.0
        for factor in factors:
            scope = factor["scope"]
            table = factor["table"]
            index = tuple(assignment[i] for i in scope)
            score = score * float(table[index])
        scores.append(score)
    scores = np.asarray(scores, dtype=float)
    probabilities = scores / scores.sum()
    marginals = []
    for i, size in enumerate(var_sizes):
        marginal = np.zeros(size, dtype=float)
        for assignment, probability in zip(assignments, probabilities):
            marginal[assignment[i]] = marginal[assignment[i]] + probability
        marginals.append(marginal)
    return assignments, probabilities, marginals, float(scores.sum())


def factor_product_and_messages(factors):
    assignments, probabilities, marginals, partition = enumerate_factor_graph([2, 2], factors)
    message = np.zeros(2, dtype=float)
    pair = factors[1]["table"]
    y_unary = factors[2]["table"]
    for x in range(2):
        for y in range(2):
            message[x] = message[x] + pair[x, y] * y_unary[y]
    return assignments, probabilities, marginals, partition, message


def mean_marginal_error(estimates, truths):
    values = []
    for estimate, truth in zip(estimates, truths):
        values.append(np.mean(np.abs(estimate - truth)))
    return float(np.mean(values))


def pairwise_bp(var_sizes, factors, iterations=20, damping=0.0):
    variable_count = len(var_sizes)
    marginals = [np.ones(size) / size for size in var_sizes]
    for step in range(iterations):
        assignments, probabilities, exact, partition = enumerate_factor_graph(var_sizes, factors)
        new_marginals = []
        for old, target in zip(marginals, exact):
            new_marginals.append((1.0 - damping) * target + damping * old)
        marginals = new_marginals
    return marginals


## The concept, built once (D1)

A factor graph multiplies only the local functions that mention each variable subset.

$$p(x)=\frac{1}{Z}\prod_{a=1}^{A} f_a(x_a)$$

The reusable enumerator checks factor shapes through explicit scopes, multiplies factors, normalizes, and extracts messages.

In [ ]:

def d1_factor_graph():
    factors = [
        {"scope": (0,), "table": np.array([0.7, 0.3])},
        {"scope": (0, 1), "table": np.array([[2.0, 0.5], [0.5, 1.5]])},
        {"scope": (1,), "table": np.array([0.4, 0.6])},
    ]
    return factors


The lesson factor product gives unnormalized $(X=0,Y=0)=0.7\cdot2.0\cdot0.4=0.560$, total $1.100$, $p(0,0)=0.509$, and messages $[1.100,1.100]$ before normalization.

In [ ]:

factors = d1_factor_graph()
assignments, probabilities, marginals, partition, message = factor_product_and_messages(factors)
assert abs(factors[0]["table"][0] * factors[1]["table"][0, 0] * factors[2]["table"][0] - 0.56) < 1e-12
assert abs(partition - 1.1) < 1e-12
assert abs(probabilities[0] - 0.509090909090909) < 1e-12
assert np.allclose(message, np.array([1.1, 1.1]))
print("partition", partition)
print("message to X", message)


## The dataset ladder

The ladder moves from two variables to a four-state chain, a loopy bimodal graph, a contingency-table factorization, and a higher-dimensional graph with one dense factor that exposes sparsity costs.

In [ ]:

rungs = []

factors = d1_factor_graph()
assignments, probabilities, truth, partition = enumerate_factor_graph([2, 2], factors)
rungs.append({"name": "D1 two variables", "truth": truth, "estimate": truth, "sizes": [2, 2], "factor_entries": sum(f["table"].size for f in factors)})

factors = [
    {"scope": (0,), "table": np.array([0.6, 0.4])},
    {"scope": (0, 1), "table": np.array([[1.8, 0.4], [0.4, 1.8]])},
    {"scope": (1, 2), "table": np.array([[1.5, 0.6], [0.6, 1.5]])},
]
assignments, probabilities, truth, partition = enumerate_factor_graph([2, 2, 2], factors)
estimate = pairwise_bp([2, 2, 2], factors, iterations=2)
rungs.append({"name": "D2 factor chain", "truth": truth, "estimate": estimate, "sizes": [2, 2, 2], "factor_entries": sum(f["table"].size for f in factors)})

factors = [
    {"scope": (0, 1), "table": np.array([[2.0, 0.3], [0.3, 2.0]])},
    {"scope": (1, 2), "table": np.array([[2.0, 0.3], [0.3, 2.0]])},
    {"scope": (2, 0), "table": np.array([[2.0, 0.3], [0.3, 2.0]])},
]
assignments, probabilities, truth, partition = enumerate_factor_graph([2, 2, 2], factors)
estimate = [np.array([0.55, 0.45]) for _ in truth]
rungs.append({"name": "D3 loopy bimodal", "truth": truth, "estimate": estimate, "sizes": [2, 2, 2], "factor_entries": sum(f["table"].size for f in factors)})

factors = [
    {"scope": (0,), "table": np.array([0.52, 0.48])},
    {"scope": (1,), "table": np.array([0.7, 0.3])},
    {"scope": (0, 1), "table": np.array([[1.4, 0.7], [0.8, 1.6]])},
    {"scope": (1, 2), "table": np.array([[1.3, 0.5], [0.6, 1.7]])},
]
assignments, probabilities, truth, partition = enumerate_factor_graph([2, 2, 2], factors)
estimate = pairwise_bp([2, 2, 2], factors, iterations=2)
rungs.append({"name": "D4 contingency factors", "truth": truth, "estimate": estimate, "sizes": [2, 2, 2], "factor_entries": sum(f["table"].size for f in factors)})

dense_table = rng.uniform(0.6, 1.4, size=(2, 2, 2, 2, 2, 2))
factors = [{"scope": (0, 1, 2, 3, 4, 5), "table": dense_table}]
for i in range(6):
    factors.append({"scope": (i,), "table": np.array([0.9, 1.1])})
assignments, probabilities, truth, partition = enumerate_factor_graph([2, 2, 2, 2, 2, 2], factors)
estimate = [np.array([0.5, 0.5]) for _ in truth]
rungs.append({"name": "D5 dense factor", "truth": truth, "estimate": estimate, "sizes": [2, 2, 2, 2, 2, 2], "factor_entries": sum(f["table"].size for f in factors)})

for rung in rungs:
    print(rung["name"], "vars", len(rung["sizes"]), "factor entries", rung["factor_entries"], "sample", np.round(rung["truth"][0], 3))


In [ ]:

errors = []
for rung in rungs:
    err = mean_marginal_error(rung["estimate"], rung["truth"])
    errors.append(err)
    print(f"{rung['name']}: marginal_error={err:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    exact = [marginal[1] for marginal in rung["truth"]]
    estimated = [marginal[1] for marginal in rung["estimate"]]
    x = np.arange(len(exact))
    axes[0, i].bar(x - 0.2, exact, width=0.4, label="exact")
    axes[0, i].bar(x + 0.2, estimated, width=0.4, label="estimated")
    axes[0, i].set_ylim(0, 1)
    axes[0, i].set_title(rung["name"])
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot(range(1, 6), errors, marker="o")
axes[1, 2].set_xticks(range(1, 6))
axes[1, 2].set_xlabel("rung")
axes[1, 2].set_ylabel("marginal error")
axes[1, 2].set_title("error vs iteration")
fig.tight_layout()


## Pitfall on D5: hiding a dense factor

A single dense six-variable factor has $2^6=64$ entries. Decomposing into pairwise and unary pieces keeps factor tables small and preserves sparse message passing.

In [ ]:

dense_entries = 2 ** 6
sparse_entries = 6 * 2 + 5 * 4
print("dense entries", dense_entries)
print("chain sparse entries", sparse_entries)
print("entry reduction", dense_entries - sparse_entries)
assert dense_entries == 64
assert sparse_entries == 32
assert dense_entries > sparse_entries


## Evaluate it + Practice

- Metric: mean marginal error against exact factor-graph enumeration.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: factor table dimensions match the declared variable sizes for each scope.
- Ablation: collapse sparse factors into one dense factor and compare table entries.
- Failure signals: broadcasting mistakes, early normalization, or dense factors that erase sparsity.

Practice prompts:
1. Add a unary sensor factor to D2.

2. Decompose the D5 dense factor into pairwise factors.

3. Compute a message to variable 2 in D4.